In [27]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [28]:
!pip3 install --upgrade datasets tensorflow keras keras-nlp keras-hub tensorflow_hub pydot graphviz numpy -q

In [29]:
import pathlib
from pathlib import Path
import random

import tensorflow.data as tf_data
import tensorflow as tf

import keras
import keras_hub

base_path = Path("./../")

data_path = base_path / "data" / "tensorflow_text"
data_path.mkdir(parents=True,exist_ok=True)
model_path = base_path / "models" / "tensorflow_text"
model_path.mkdir(parents=True,exist_ok=True)

In [30]:
model_path.resolve()

PosixPath('/Users/cerozob/Desktop/MAIA-Sagemaker/models/tensorflow_text')

Lo siguiente es seguir este tutorial: https://keras.io/examples/nlp/neural_machine_translation_with_transformer

los datos

In [31]:
text_file = keras.utils.get_file(
    fname="spa-eng.zip",
    origin="http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip",
    extract=True,
    cache_dir=data_path
)
text_file = pathlib.Path(text_file).parent / "spa-eng_extracted" / "spa-eng"/ "spa.txt"

parsing de los datos

In [32]:
with open(text_file) as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    eng, spa = line.split("\t")
    text_pairs.append({
        "english":eng,
        "spanish":spa
    })

for _ in range(3):
    print(random.choice(text_pairs))


{'english': 'Tom handed a note to Mary.', 'spanish': 'Tom le entregó una nota a Mary.'}
{'english': 'The boy standing by the door is my brother.', 'spanish': 'El chico parado en la puerta es mi hermano.'}
{'english': 'I hear the drum.', 'spanish': 'Oigo los tambores.'}


Preprocesar texto

shuffle and split


In [33]:
random.shuffle(text_pairs)
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples : num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples :]

print(f"{len(text_pairs)} total pairs")
print(f"{len(train_pairs)} training pairs")
print(f"{len(val_pairs)} validation pairs")
print(f"{len(test_pairs)} test pairs")

118964 total pairs
83276 training pairs
17844 validation pairs
17844 test pairs


ya hay un preprocessor, asì que podemos crear el dataset usando strings

construir el encoder

In [34]:
bert_backbone = keras_hub.models.BertBackbone.from_preset("bert_base_multi")
bert_preprocessor = keras_hub.models.BertPreprocessor.from_preset("bert_base_multi")

In [35]:
import numpy as np
BATCH_SIZE=16

def preprocess_batch(src_lang, dst_lang):
    dst_lang,label = dst_lang
    label = tf.expand_dims(label, axis=1)
    dst_preprocessed = bert_preprocessor(dst_lang)
    src_preprocessed = bert_preprocessor(src_lang)

    dst_tokenids = dst_preprocessed["token_ids"]

    return (
        (
            src_preprocessed["token_ids"],
            src_preprocessed["padding_mask"],
            src_preprocessed["segment_ids"],
            dst_tokenids[:, :-1],
        ),
        (dst_tokenids[:, 1:], label),
    )


def make_dataset_bid(pairs):
    eng_texts = list(pair["english"] for pair in pairs)
    spa_texts = list(pair["spanish"] for pair in pairs)
    labels = list(0 for _ in range(len(pairs)))
    flippedcounter = 0
    for i in range(len(pairs)):
        if np.random.rand() > 0.5:
            eng_texts[i], spa_texts[i] = spa_texts[i], eng_texts[i]
            labels[i] = 1
            flippedcounter += 1
    print(f"flipped {flippedcounter} pairs, {flippedcounter/len(pairs)}%")
    dataset1 = tf_data.Dataset.from_tensor_slices((spa_texts, eng_texts, labels)) # 1 means source language was spanish
    dataset1 = dataset1.map(lambda es,en,la: (es,(en,la)))

    # dataset2 = tf_data.Dataset.from_tensor_slices((eng_texts, (spa_texts,labels))) # 0 means source language was english
    # dataset2 = dataset2.map(lambda en,es: (en,(es,0)))
    
    # dataset = dataset1.concatenate(dataset2).batch(BATCH_SIZE)
    dataset = dataset1.batch(BATCH_SIZE)
    dataset = dataset.map(preprocess_batch,num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.shuffle(BATCH_SIZE*10
                           ).prefetch(tf.data.AUTOTUNE).cache()

train_ds = make_dataset_bid(train_pairs)
val_ds = make_dataset_bid(val_pairs)

flipped 41565 pairs, 0.4991233968970652%
flipped 8799 pairs, 0.4931069266980498%


In [36]:
text_preprocessed = bert_preprocessor(["hola"])

print(f'Keys       : {list(text_preprocessed.keys())}')
print(f'Shape      : {text_preprocessed["token_ids"].shape}')
print(f'Word Ids   : {text_preprocessed["token_ids"][0, :12]}')
print(f'Input Mask : {text_preprocessed["padding_mask"][0, :12]}')
print(f'Type Ids   : {text_preprocessed["segment_ids"][0, :12]}')

Keys       : ['token_ids', 'padding_mask', 'segment_ids']
Shape      : (1, 512)
Word Ids   : [   101 110516  10113    102      0      0      0      0      0      0
      0      0]
Input Mask : [ True  True  True  True False False False False False False False False]
Type Ids   : [0 0 0 0 0 0 0 0 0 0 0 0]


In [37]:
bertconf = bert_backbone.get_config()
print(bertconf)
SPA_VOCAB_SIZE = bertconf["vocabulary_size"]
VOCAB_SIZE = bertconf["vocabulary_size"]
MAX_SEQUENCE_LENGTH = bertconf["max_sequence_length"]
EMBED_DIM = bertconf["hidden_dim"]
INTERMEDIATE_DIM = bertconf["intermediate_dim"]
NUM_HEADS_BERT = bertconf["num_heads"]

{'name': 'bert_backbone', 'trainable': True, 'vocabulary_size': 119547, 'num_layers': 12, 'num_heads': 12, 'hidden_dim': 768, 'intermediate_dim': 3072, 'dropout': 0.1, 'max_sequence_length': 512, 'num_segments': 2}


In [38]:
bert_preprocessor.trainable=False
bert_preprocessor.name="Preprocessor_BERT"
bert_backbone.trainable=False
bert_backbone.name="Encoder_BERT"
bert_preprocessor.get_config()

{'name': 'Preprocessor_BERT',
 'trainable': False,
 'dtype': {'module': 'keras',
  'class_name': 'DTypePolicy',
  'config': {'name': 'float32'},
  'registered_name': None},
 'tokenizer': {'module': 'keras_hub.src.models.bert.bert_tokenizer',
  'class_name': 'BertTokenizer',
  'config': {'name': 'bert_tokenizer',
   'trainable': False,
   'dtype': {'module': 'keras',
    'class_name': 'DTypePolicy',
    'config': {'name': 'int32'},
    'registered_name': None},
   'config_file': 'tokenizer.json',
   'vocabulary': None,
   'sequence_length': None,
   'lowercase': False,
   'strip_accents': False,
   'split': True,
   'suffix_indicator': '##',
   'oov_token': '[UNK]',
   'special_tokens': None,
   'special_tokens_in_strings': False},
  'registered_name': 'keras_hub>BertTokenizer'},
 'config_file': 'preprocessor.json',
 'sequence_length': 512,
 'truncate': 'round_robin'}

In [39]:
# text_input = keras.layers.Input(shape=(), dtype=string, name='Text_Input_Layer')
# encoder_inputs = bert_preprocessor(text_input)


# encoder_inputs["token_ids"].name = "token_ids"
# encoder_inputs["padding_mask"].name = "padding_mask"
# encoder_inputs["segment_ids"].name = "segment_ids"

# encoder_outputs = bert_backbone(encoder_inputs)
# encoder_outputs_seq = encoder_outputs['sequence_output']
# encoder_outputs_pool = encoder_outputs['pooled_output']

# encoder = keras.Model(text_input, encoder_outputs_seq,name="Bert-multi-encoder")


# encoder.trainable = False

# Create inputs for tokenized data
token_ids_input = keras.layers.Input(
    shape=(None,), dtype=tf.int32, name="token_ids_input"
)
padding_mask_input = keras.layers.Input(
    shape=(None,), dtype=tf.bool, name="padding_mask_input"
)
segment_ids_input = keras.layers.Input(
    shape=(None,), dtype=tf.int32, name="segment_ids_input"
)

# Create a dictionary of inputs that the BERT backbone expects
encoder_inputs = {
    "token_ids": token_ids_input,
    "padding_mask": padding_mask_input,
    "segment_ids": segment_ids_input,
}

# Pass these inputs directly to the BERT backbone
encoder_outputs = bert_backbone(encoder_inputs)
encoder_outputs_seq = encoder_outputs["sequence_output"]
encoder_outputs_pool = encoder_outputs["pooled_output"]

# Create the encoder model with the new inputs
encoder = keras.Model(
    [token_ids_input, padding_mask_input, segment_ids_input],
    encoder_outputs_seq,
    name="Bert-multi-encoder",
)

encoder.summary(show_trainable=True)

Model: "Bert-multi-encoder"

┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)      ┃ Output Shape    ┃   Param # ┃ Connected to   ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ padding_mask_inp… │ (None, None)    │         0 │ -              │   -   │
│ (InputLayer)      │                 │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ segment_ids_input │ (None, None)    │         0 │ -              │   -   │
│ (InputLayer)      │                 │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ token_ids_input   │ (None, None)    │         0 │ -              │   -   │
│ (InputLayer)      │                 │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ Encoder_BERT      │ [(None, 768),   │ 177,853,… │ padding_mask_… │   N   │
│ (BertBackbone)    │ (None, None,    │           │ segment_ids_i… │       │
│                   │ 768)]           │           │ token_ids_inp… │       │
└───────────────────┴─────────────────┴───────────┴────────────────┴───────┘

 Total params: 177,853,440 (678.46 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 177,853,440 (678.46 MB)

In [40]:
# keras.utils.plot_model(encoder, show_shapes=True, show_layer_names=True,show_trainable=True)

decoder

In [41]:
REDUCED_EMBED_DIM = EMBED_DIM//4
REDUCED_INTERMEDIATE_DIM = REDUCED_EMBED_DIM*4
SHORTENED_SEQUENCE_LENGTH = MAX_SEQUENCE_LENGTH
REDUCED_HEADS= 6

# Decoder
decoder_inputs = keras.Input(shape=(None,), name="decoder_inputs")
encoded_seq_inputs = keras.Input(shape=(None, EMBED_DIM), name="decoder_state_inputs")


shared_embedding = keras_hub.layers.TokenAndPositionEmbedding(
    vocabulary_size=SPA_VOCAB_SIZE,
    sequence_length=SHORTENED_SEQUENCE_LENGTH,
    embedding_dim=REDUCED_EMBED_DIM,
)(decoder_inputs)


shared_decoder = keras_hub.layers.TransformerDecoder(
    intermediate_dim=REDUCED_INTERMEDIATE_DIM, num_heads=REDUCED_HEADS
)(decoder_sequence=shared_embedding, encoder_sequence=encoded_seq_inputs)

shared_decoder = keras.layers.Dropout(0.7)(shared_embedding)


decoder_outputs = keras.layers.Dense(
    SPA_VOCAB_SIZE, activation="softmax", name="translation_output"
)(shared_decoder)
decoder = keras.Model(
    [
        decoder_inputs,
        encoded_seq_inputs,
    ],
    decoder_outputs,
    name="TranslationDecoder"
)
decoder_outputs = decoder([decoder_inputs, encoder_outputs_seq])

decoder.summary(show_trainable=True,line_length=200)

Model: "TranslationDecoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━
┃ Layer (type)                                        ┃ Output Shape                                ┃              
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━
│ decoder_inputs (InputLayer)                         │ (None, None)                                │              
├─────────────────────────────────────────────────────┼─────────────────────────────────────────────┼──────────────
│ token_and_position_embedding_3                      │ (None, None, 192)                           │              
│ (TokenAndPositionEmbedding)                         │                                             │              
├─────────────────────────────────────────────────────┼─────────────────────────────────────────────┼──────────────
│ dropout_59 (Dropout)                                │ (None, None, 192)                           │              
├─────────────────────────────────────────────────────┼─────────────────────────────────────────────┼──────────────
│ decoder_state_inputs (InputLayer)                   │ (None, None, 768)                           │              
├─────────────────────────────────────────────────────┼─────────────────────────────────────────────┼──────────────
│ translation_output (Dense)                          │ (None, None, 119547)                        │              
└─────────────────────────────────────────────────────┴─────────────────────────────────────────────┴──────────────

 Total params: 46,123,899 (175.95 MB)

 Trainable params: 46,123,899 (175.95 MB)

 Non-trainable params: 0 (0.00 B)

In [48]:
# keras.utils.plot_model(    decoder, show_shapes=True, show_layer_names=True, show_trainable=True)

In [ ]:
pooled_output = keras.layers.GlobalAveragePooling1D()(shared_decoder)

decoder_classification_output = keras.layers.Dense(units=2, activation="softmax",name="classification_output")(pooled_output)

decoder_classification = keras.Model(
    [
        decoder_inputs,
        encoded_seq_inputs
    ],
    decoder_classification_output,
    name="LanguageIdentificationDecoder"
)


decoder_classification_outputs = decoder_classification(
    [decoder_inputs, encoder_outputs_seq]
)
decoder_classification.summary(line_length=100)

Model: "LanguageIdentificationDecoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_and_position… │ (None, None, 192) │ 23,051,328 │ decoder_inputs[0… │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_77          │ (None, None, 192) │          0 │ token_and_positi… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 192)       │          0 │ dropout_77[0][0]  │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_state_inpu… │ (None, None, 768) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classification_out… │ (None, 2)         │        386 │ global_average_p… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,051,714 (87.94 MB)

 Trainable params: 23,051,714 (87.94 MB)

 Non-trainable params: 0 (0.00 B)

In [50]:
# keras.utils.plot_model(    decoder_classification, show_shapes=True, show_layer_names=True, show_trainable=True)

Componer el modelo completo

In [51]:
# transformer = keras.Model(
#     [text_input,decoder_inputs],
#     [decoder_outputs,decoder_classification_outputs],
#     name="TranslationTransformer",
# )
transformer = keras.Model(
    [token_ids_input, padding_mask_input, segment_ids_input, decoder_inputs],
    [decoder_outputs, decoder_classification_outputs],
    name="TranslationTransformer",
)

transformer.summary(show_trainable=True, line_length=100)

Model: "TranslationTransformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Layer (type)             ┃ Output Shape         ┃       Param # ┃ Connected to         ┃ Traina… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ padding_mask_input       │ (None, None)         │             0 │ -                    │    -    │
│ (InputLayer)             │                      │               │                      │         │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ segment_ids_input        │ (None, None)         │             0 │ -                    │    -    │
│ (InputLayer)             │                      │               │                      │         │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ token_ids_input          │ (None, None)         │             0 │ -                    │    -    │
│ (InputLayer)             │                      │               │                      │         │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ decoder_inputs           │ (None, None)         │             0 │ -                    │    -    │
│ (InputLayer)             │                      │               │                      │         │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ Encoder_BERT             │ [(None, 768), (None, │   177,853,440 │ padding_mask_input[… │    N    │
│ (BertBackbone)           │ None, 768)]          │               │ segment_ids_input[0… │         │
│                          │                      │               │ token_ids_input[0][… │         │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ TranslationDecoder       │ (None, None, 119547) │    46,123,899 │ decoder_inputs[0][0… │    Y    │
│ (Functional)             │                      │               │ Encoder_BERT[0][1]   │         │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ LanguageIdentificationD… │ (None, 2)            │    23,051,714 │ decoder_inputs[0][0… │    Y    │
│ (Functional)             │                      │               │ Encoder_BERT[0][1]   │         │
└──────────────────────────┴──────────────────────┴───────────────┴──────────────────────┴─────────┘

 Total params: 223,977,725 (854.41 MB)

 Trainable params: 46,124,285 (175.95 MB)

 Non-trainable params: 177,853,440 (678.46 MB)

In [19]:
# keras.utils.plot_model(    transformer, show_shapes=True, show_layer_names=True, show_trainable=True)

a entrenar!

In [25]:
with tf.device('/GPU:0'):

    # set up an scheduler to fine tune rather quickly
    exponential_decay = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=0.1,
        decay_steps=10000,
        decay_rate=0.9)

    optimizer = keras.optimizers.Adam(learning_rate=exponential_decay)

    early_stopping = keras.callbacks.EarlyStopping(
        patience=5,
        min_delta=0.01,
        restore_best_weights=True
    )
    # there are 5K datapoints, we want to set the steps_per_epoch and epochs to account for one full pass
    # calculate everything in function of BATCH_SIZE and each dataset
   
    epochs = 100
    train_steps = len(train_ds) // epochs 
    val_steps = len(val_ds) // epochs

    transformer.compile(
        optimizer=optimizer,
            loss={
            "TranslationDecoder": "sparse_categorical_crossentropy",
            "LanguageIdentificationDecoder": "sparse_categorical_crossentropy"
        }, metrics={
            "TranslationDecoder": "accuracy",  #keras_hub.metrics.Bleu,
            "LanguageIdentificationDecoder": "categorical_accuracy"
        },
        
        jit_compile=False
    )
    transformer.fit(train_ds,
                    epochs=epochs, 
                    steps_per_epoch=train_steps,
                    validation_steps=val_steps, 
                    callbacks=[early_stopping],
                    validation_data=val_ds)

Epoch 1/100


2025-05-03 06:52:09.758470: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2025-05-03 06:52:09.916079: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


52/52 ━━━━━━━━━━━━━━━━━━━━ 40s 667ms/step - LanguageIdentificationDecoder_categorical_accuracy: 0.4595 - LanguageIdentificationDecoder_loss: 15.7319 - TranslationDecoder_accuracy: 0.9812 - TranslationDecoder_loss: 2.2740 - loss: 18.0059 - val_LanguageIdentificationDecoder_categorical_accuracy: 1.0000 - val_LanguageIdentificationDecoder_loss: 0.9102 - val_TranslationDecoder_accuracy: 0.9862 - val_TranslationDecoder_loss: 1.2812 - val_loss: 2.1914
Epoch 2/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step - LanguageIdentificationDecoder_categorical_accuracy: 0.4479 - LanguageIdentificationDecoder_loss: 1.1274 - TranslationDecoder_accuracy: 0.9839 - TranslationDecoder_loss: 1.8484 - loss: 2.9758

2025-05-03 06:53:13.197462: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


52/52 ━━━━━━━━━━━━━━━━━━━━ 30s 585ms/step - LanguageIdentificationDecoder_categorical_accuracy: 0.4484 - LanguageIdentificationDecoder_loss: 1.1253 - TranslationDecoder_accuracy: 0.9839 - TranslationDecoder_loss: 1.8501 - loss: 2.9754 - val_LanguageIdentificationDecoder_categorical_accuracy: 1.0000 - val_LanguageIdentificationDecoder_loss: 0.7547 - val_TranslationDecoder_accuracy: 0.9856 - val_TranslationDecoder_loss: 1.4762 - val_loss: 2.2309
Epoch 3/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step - LanguageIdentificationDecoder_categorical_accuracy: 0.4964 - LanguageIdentificationDecoder_loss: 0.9900 - TranslationDecoder_accuracy: 0.9840 - TranslationDecoder_loss: 1.8820 - loss: 2.8719

...

2025-05-03 06:56:16.287767: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


52/52 ━━━━━━━━━━━━━━━━━━━━ 30s 585ms/step - LanguageIdentificationDecoder_categorical_accuracy: 0.5658 - LanguageIdentificationDecoder_loss: 1.5922 - TranslationDecoder_accuracy: 0.9830 - TranslationDecoder_loss: 2.1749 - loss: 3.7671 - val_LanguageIdentificationDecoder_categorical_accuracy: 0.0000e+00 - val_LanguageIdentificationDecoder_loss: 0.9823 - val_TranslationDecoder_accuracy: 0.9863 - val_TranslationDecoder_loss: 1.4626 - val_loss: 2.4449
Epoch 9/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step - LanguageIdentificationDecoder_categorical_accuracy: 0.4182 - LanguageIdentificationDecoder_loss: 1.1487 - TranslationDecoder_accuracy: 0.9832 - TranslationDecoder_loss: 2.1215 - loss: 3.2702

2025-05-03 06:56:46.625156: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


52/52 ━━━━━━━━━━━━━━━━━━━━ 30s 584ms/step - LanguageIdentificationDecoder_categorical_accuracy: 0.4192 - LanguageIdentificationDecoder_loss: 1.1461 - TranslationDecoder_accuracy: 0.9832 - TranslationDecoder_loss: 2.1209 - loss: 3.2671 - val_LanguageIdentificationDecoder_categorical_accuracy: 1.0000 - val_LanguageIdentificationDecoder_loss: 1.0444 - val_TranslationDecoder_accuracy: 0.9856 - val_TranslationDecoder_loss: 1.4729 - val_loss: 2.5173


2025-05-03 06:56:51.901420: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2025-05-03 06:56:52.728156: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [52]:
bert_tokenizer = keras_hub.models.Tokenizer.from_preset("bert_base_multi")

In [53]:
evaluation_results = transformer.evaluate(val_ds, steps=val_steps)
print(f"Evaluation Results: {evaluation_results}")

NameError: name 'val_steps' is not defined

In [76]:
def translate_with_greedy_search(
    input_text, model=None, max_length=50):
    """
    Translate text using greedy decoding and identify its language.

    Args:
        input_text: Text to translate
        model: The transformer model to use
        max_length: Maximum length of generated translation
        show_special_tokens: Whether to show special tokens in the output

    Returns:
        Dictionary with translation results
    """
    # Get the BERT tokenizer
    bert_tokenizer = keras_hub.models.Tokenizer.from_preset("bert_base_multi")

    # Preprocess the input text using the BERT preprocessor
    preprocessed = bert_preprocessor([input_text])

    # Extract tokenized inputs
    token_ids = preprocessed["token_ids"]
    padding_mask = preprocessed["padding_mask"]
    segment_ids = preprocessed["segment_ids"]

    # Initialize with start token
    start_token_id = bert_tokenizer.token_to_id("[CLS]")
    end_token_id = bert_tokenizer.token_to_id("[SEP]")

    # Create initial decoder input with start token
    decoder_input = tf.constant([[start_token_id]])

    # Identify the language first
    _, classification_output = model.predict(
        [token_ids, padding_mask, segment_ids, decoder_input], verbose=0
    )

    language_id = tf.argmax(classification_output[0]).numpy()
    language = "English" if language_id == 0 else "Spanish"

    # Generate translation token by token using greedy search
    generated_tokens = [start_token_id]
    token_probabilities = []

    for i in range(max_length):
        # Prepare decoder input from generated tokens so far
        decoder_input = tf.constant([generated_tokens])

        # Get model prediction
        translation_output, _ = model.predict(
            [token_ids, padding_mask, segment_ids, decoder_input], verbose=0
        )

        # Get the next token prediction (last position in sequence)
        next_token_logits = translation_output[0, -1, :]
        next_token_probs = tf.nn.softmax(next_token_logits).numpy()

        # Greedy selection - pick the most probable token
        next_token_id = int(np.argmax(next_token_probs))
        next_token_prob = float(next_token_probs[next_token_id])

        # Add the predicted token and its probability
        generated_tokens.append(next_token_id)
        token_probabilities.append(next_token_prob)

        # Stop if we predict the end token
        if next_token_id == end_token_id:
            break

    # Process the generated tokens
    token_strings = []
    for token_id in generated_tokens:
        token = bert_tokenizer.id_to_token(token_id)
        token_strings.append(token)

    # Create the translation text
    translation = " ".join(token_strings)
    clean_translation = filtered_tokens = [
            t for t in generated_tokens if t not in [start_token_id, end_token_id]
        ]
    clean_translation = bert_tokenizer.detokenize(filtered_tokens)
   
    # Calculate overall translation confidence
    avg_confidence = (
        sum(token_probabilities) / len(token_probabilities)
        if token_probabilities
        else 0
    )

    return {
        "language": language,
        "original_text": input_text,
        "translation": translation,
        "clean_translation": clean_translation,
        "tokens": token_strings,
        "token_ids": generated_tokens,
        "confidence": float(avg_confidence),
    }

input_text = "Hi, this is a sample sentence."
result = translate_with_greedy_search(input_text)

print(f"Input: {input_text}")
print(f"Identified Language: {result['language']}")
print(f"Original Text: {result['original_text']}")
print(f"Translation: {result['translation']}")
print(f"Clean Translation: {result['clean_translation']}")
print(f"Overall confidence: {result['confidence']:.4f}")


AttributeError: 'NoneType' object has no attribute 'predict'

In [ ]:
from keras_hub.layers import PositionEmbedding, TransformerDecoder

transformer.save(
    model_path / "translation_transformer.keras",
    overwrite=True,
    include_optimizer=False
)


In [ ]:
# 1. Save only the weights instead of the full model
transformer.save_weights(model_path / "transformer.weights.h5")

In [1]:
!pip install --upgrade keras_hub keras

In [4]:
import keras
from keras import backend as K

print(K.backend())

tensorflow


In [61]:
import tensorflow as tf

keras.__version__

'3.9.2'

In [72]:
transformer4 = keras.models.load_model(
    model_path / "translation_transformer.keras",
    custom_objects={
        "PositionEmbedding": keras_hub.layers.PositionEmbedding,
        "ReversibleEmbedding":keras_hub.layers.ReversibleEmbedding
    },
)

/Users/cerozob/Desktop/MAIA-Sagemaker/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 17 variables whereas the saved optimizer has 13 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


ValueError: A total of 2 objects could not be loaded. Example error message for object <PositionEmbedding name=position_embedding, built=True>:

Layer 'position_embedding' expected 1 variables, but received 0 variables during loading. Expected: ['embeddings']

List of objects that could not be loaded:
[<PositionEmbedding name=position_embedding, built=True>, <ReversibleEmbedding name=token_embedding, built=True>]

In [70]:
import keras_hub
import keras
import tensorflow as tf
# 2. Create a function to build the model with the same architecture
def create_transformer_model():
    # Get the BERT tokenizer and preprocessor
    bert_preprocessor = keras_hub.models.BertPreprocessor.from_preset("bert_base_multi")
    bert_backbone = keras_hub.models.BertBackbone.from_preset("bert_base_multi")
    bert_tokenizer = bert_preprocessor.tokenizer
    # Get configuration values
    bertconf = bert_backbone.get_config()
    VOCAB_SIZE = bertconf["vocabulary_size"]
    MAX_SEQUENCE_LENGTH = bertconf["max_sequence_length"]
    EMBED_DIM = bertconf["hidden_dim"]
    INTERMEDIATE_DIM = bertconf["intermediate_dim"]
    NUM_HEADS_BERT = bertconf["num_heads"]

    # Set up the encoder
    bert_preprocessor.trainable = False
    bert_preprocessor.name = "Preprocessor_BERT"
    bert_backbone.trainable = False
    bert_backbone.name = "Encoder_BERT"

    # Create inputs for tokenized data
    token_ids_input = keras.layers.Input(
        shape=(None,), dtype=tf.int32, name="token_ids_input"
    )
    padding_mask_input = keras.layers.Input(
        shape=(None,), dtype=tf.bool, name="padding_mask_input"
    )
    segment_ids_input = keras.layers.Input(
        shape=(None,), dtype=tf.int32, name="segment_ids_input"
    )

    # Create a dictionary of inputs that the BERT backbone expects
    encoder_inputs = {
        "token_ids": token_ids_input,
        "padding_mask": padding_mask_input,
        "segment_ids": segment_ids_input,
    }

    # Pass these inputs directly to the BERT backbone
    encoder_outputs = bert_backbone(encoder_inputs)
    encoder_outputs_seq = encoder_outputs["sequence_output"]

    # Create the encoder model
    encoder = keras.Model(
        [token_ids_input, padding_mask_input, segment_ids_input],
        encoder_outputs_seq,
        name="Bert-multi-encoder",
    )

    # Decoder parameters
    REDUCED_EMBED_DIM = EMBED_DIM // 4
    REDUCED_INTERMEDIATE_DIM = REDUCED_EMBED_DIM * 4
    SHORTENED_SEQUENCE_LENGTH = MAX_SEQUENCE_LENGTH
    REDUCED_HEADS = 6

    # Decoder
    decoder_inputs = keras.Input(shape=(None,), name="decoder_inputs")

    # Token and position embedding
    shared_embedding = keras_hub.layers.TokenAndPositionEmbedding(
        vocabulary_size=VOCAB_SIZE,
        sequence_length=SHORTENED_SEQUENCE_LENGTH,
        embedding_dim=REDUCED_EMBED_DIM,
    )(decoder_inputs)

    # Dropout layer
    shared_decoder = keras.layers.Dropout(0.7)(shared_embedding)

    # Translation output layer
    decoder_outputs = keras.layers.Dense(
        VOCAB_SIZE, activation="softmax", name="translation_output"
    )(shared_decoder)

    # Create decoder model
    decoder = keras.Model(
        [decoder_inputs, encoder_outputs_seq],
        decoder_outputs,
        name="TranslationDecoder",
    )

    decoder_outputs = decoder([decoder_inputs, encoder_outputs_seq])

    # Language identification
    pooled_output = keras.layers.GlobalAveragePooling1D()(shared_decoder)
    decoder_classification_output = keras.layers.Dense(
        units=2, activation="softmax", name="classification_output"
    )(pooled_output)

    # Create language identification model
    decoder_classification = keras.Model(
        [decoder_inputs, encoder_outputs_seq],
        decoder_classification_output,
        name="LanguageIdentificationDecoder",
    )

    decoder_classification_outputs = decoder_classification(
        [decoder_inputs, encoder_outputs_seq]
    )

    # Create the full transformer model
    transformer = keras.Model(
        [token_ids_input, padding_mask_input, segment_ids_input, decoder_inputs],
        [decoder_outputs, decoder_classification_outputs],
        name="TranslationTransformer",
    )
    return transformer


# 3. Load the saved weights into the recreated model
transformer2 = create_transformer_model()
transformer2.load_weights(model_path / "transformer.weights.h5")
transformer2.compile(jit_compile=False)

In [75]:
transformer2.export(
    model_path/"tf_exported_model"
)

INFO:tensorflow:Assets written to: ../models/tensorflow_text/tf_exported_model/assets


INFO:tensorflow:Assets written to: ../models/tensorflow_text/tf_exported_model/assets


Saved artifact at '../models/tensorflow_text/tf_exported_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, None), dtype=tf.int32, name='token_ids_input'), TensorSpec(shape=(None, None), dtype=tf.bool, name='padding_mask_input'), TensorSpec(shape=(None, None), dtype=tf.int32, name='segment_ids_input'), TensorSpec(shape=(None, None), dtype=tf.float32, name='decoder_inputs')]
Output Type:
  List[TensorSpec(shape=(None, None, 119547), dtype=tf.float32, name=None), TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)]
Captures:
  6131771984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6131773712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6131774096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6131772560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6131772752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6131774672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6

In [77]:
def translate_with_greedy_search_savedmodel(input_text, model=None, max_length=50):
    """
    Translate text using greedy decoding and identify its language.

    Args:
        input_text: Text to translate
        model: The transformer model to use
        max_length: Maximum length of generated translation
        show_special_tokens: Whether to show special tokens in the output

    Returns:
        Dictionary with translation results
    """
    # Get the BERT tokenizer
    bert_tokenizer = keras_hub.models.Tokenizer.from_preset("bert_base_multi")

    # Preprocess the input text using the BERT preprocessor
    preprocessed = bert_preprocessor([input_text])

    # Extract tokenized inputs
    token_ids = preprocessed["token_ids"]
    padding_mask = preprocessed["padding_mask"]
    segment_ids = preprocessed["segment_ids"]

    # Initialize with start token
    start_token_id = bert_tokenizer.token_to_id("[CLS]")
    end_token_id = bert_tokenizer.token_to_id("[SEP]")

    # Create initial decoder input with start token
    decoder_input = tf.constant([[start_token_id]])

    # Identify the language first
    _, classification_output = model.serve(
        [token_ids, padding_mask, segment_ids, decoder_input]
    )

    language_id = tf.argmax(classification_output[0]).numpy()
    language = "English" if language_id == 0 else "Spanish"

    # Generate translation token by token using greedy search
    generated_tokens = [start_token_id]
    token_probabilities = []

    for i in range(max_length):
        # Prepare decoder input from generated tokens so far
        decoder_input = tf.constant([generated_tokens])

        # Get model prediction
        translation_output, _ = model.serve(
            [token_ids, padding_mask, segment_ids, decoder_input]
        )

        # Get the next token prediction (last position in sequence)
        next_token_logits = translation_output[0, -1, :]
        next_token_probs = tf.nn.softmax(next_token_logits).numpy()

        # Greedy selection - pick the most probable token
        next_token_id = int(np.argmax(next_token_probs))
        next_token_prob = float(next_token_probs[next_token_id])

        # Add the predicted token and its probability
        generated_tokens.append(next_token_id)
        token_probabilities.append(next_token_prob)

        # Stop if we predict the end token
        if next_token_id == end_token_id:
            break

    # Process the generated tokens
    token_strings = []
    for token_id in generated_tokens:
        token = bert_tokenizer.id_to_token(token_id)
        token_strings.append(token)

    # Create the translation text
    translation = " ".join(token_strings)
    clean_translation = filtered_tokens = [
        t for t in generated_tokens if t not in [start_token_id, end_token_id]
    ]
    clean_translation = bert_tokenizer.detokenize(filtered_tokens)

    # Calculate overall translation confidence
    avg_confidence = (
        sum(token_probabilities) / len(token_probabilities)
        if token_probabilities
        else 0
    )

    return {
        "language": language,
        "original_text": input_text,
        "translation": translation,
        "clean_translation": clean_translation,
        "tokens": token_strings,
        "token_ids": generated_tokens,
        "confidence": float(avg_confidence),
    }

In [80]:
tf3 = tf.saved_model.load(model_path / "tf_exported_model")


In [81]:
# Example usage
input_text = "Hi, this is a sample sentence."

result = translate_with_greedy_search_savedmodel(input_text, model=tf3)

TypeError: Binding inputs to tf.function failed due to `Can not cast TensorSpec(shape=(1, 1), dtype=tf.int32, name='decoder_inputs') to TensorSpec(shape=(None, None), dtype=tf.float32, name='decoder_inputs')`. Received args: ([<tf.Tensor: shape=(1, 512), dtype=int32, numpy=
array([[  101, 20065,   117, 10531, 10124,   169, 45700, 49219,   119,
          102,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0, 

In [57]:
result

{'language': 'Spanish',
 'original_text': 'Hi, this is a sample sentence.',
 'translation': "[CLS] I ' s a la biblioteca . [SEP]",
 'clean_translation': "I ' s a la biblioteca .",
 'tokens': ['[CLS]', 'I', "'", 's', 'a', 'la', 'biblioteca', '.', '[SEP]'],
 'token_ids': [101, 146, 112, 187, 169, 10109, 34297, 119, 102],
 'confidence': 2.052473462299531e-05}

### sagemaker inference script

In [63]:
import json
result.pop("tokens")
result.pop("token_ids")
json.dumps(result)

'{"language": "Spanish", "original_text": "Hi, this is a sample sentence.", "translation": "[CLS] I \' s a la biblioteca . [SEP]", "clean_translation": "I \' s a la biblioteca .", "confidence": 2.052473462299531e-05}'